In [ ]:
# Main imports
from torchvision import models, datasets, transforms
import torch as t
from dataclasses import dataclass
from torch.utils.data import Subset, DataLoader
import torch.nn.functional as F
from tqdm.notebook import tqdm

In [ ]:
# Local imports: this cell imports from the files, either run this
# or copy and run the file contents directly
from models._resnet34 import ResNet34
from models._resnet_components import Linear

In [ ]:
device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")

In [ ]:
# Copy weights from Torch's trained imagenet resnet
########################################################################
def copy_weights(my_resnet: ResNet34, pretrained_resnet: models.resnet.ResNet) -> ResNet34:
    """Copy over the weights of `pretrained_resnet` to your resnet."""

    # Get the state dictionaries for each model, check they have the same number of parameters &
    # buffers
    mydict = my_resnet.state_dict()
    pretraineddict = pretrained_resnet.state_dict()
    assert len(mydict) == len(pretraineddict), "Mismatching state dictionaries."

    # Define a dictionary mapping the names of your parameters / buffers to their values in the
    # pretrained model
    state_dict_to_load = {
        mykey: pretrainedvalue
        for (mykey, myvalue), (pretrainedkey, pretrainedvalue) in zip(mydict.items(), pretraineddict.items())
    }

    # Load in this dictionary to your model
    my_resnet.load_state_dict(state_dict_to_load)
    print("Weights copied successfully!")

    return my_resnet

In [ ]:
# Freeze model, change last layer to new n_classes to be trained
########################################################################
def get_resnet_for_feature_extraction(n_classes: int) -> ResNet34:
    """
    Creates a ResNet34 instance, replaces its final linear layer with a classifier for `n_classes`
    classes, and freezes all weights except the ones in this layer.

    Returns the ResNet model.
    """
    # Create model instance and copy weights
    model = ResNet34()
    pretrained_resnet = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)
    model = copy_weights(model, pretrained_resnet)

    # Freeze all layers
    model.requires_grad_(False)

    # Replace final layer
    model.out_layers[-1] = Linear(model.out_features_per_group[-1], n_classes)

    print("Resnet prepared successfully")
    return model

In [ ]:
# Images transform
#####################################################
IMAGE_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

IMAGENET_TRANSFORM = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)

In [ ]:
# Load CIFAR dataset
#####################################################
def colab_get_cifar() -> tuple[datasets.CIFAR10, datasets.CIFAR10]:
    """Returns CIFAR-10 train and test sets."""
    from google.colab import drive
    drive.mount('/content/drive')

    !mkdir -p /content/data
    !cp "/content/drive/MyDrive/data/cifar-10-python.tar.gz" /content/data/

    cifar_trainset = datasets.CIFAR10("/content/data", train=True, download=True, transform=IMAGENET_TRANSFORM)
    cifar_testset = datasets.CIFAR10("/content/data", train=False, download=True, transform=IMAGENET_TRANSFORM)
    return cifar_trainset, cifar_testset

def get_cifar_subset(trainset_size: int = 10_000, testset_size: int = 1_000) -> tuple[Subset, Subset]:
    """Returns a subset of CIFAR-10 train & test sets (slicing the first examples)."""
    cifar_trainset, cifar_testset = colab_get_cifar()
    return Subset(cifar_trainset, range(trainset_size)), Subset(cifar_testset, range(testset_size))


In [ ]:
# Training args
#####################################################
@dataclass
class ResNetTrainingArgs:
    batch_size: int = 64
    epochs: int = 5
    learning_rate: float = 1e-3
    n_classes: int = 10

In [ ]:
# Train
#####################################################
def train(args: ResNetTrainingArgs) -> tuple[list[float], list[float], ResNet34]:
    """
    Performs feature extraction on ResNet, returning the model & lists of loss and accuracy.
    """
    # Create ResNet instance
    model = get_resnet_for_feature_extraction(args.n_classes).to(device)
    # Get data
    cifar_trainset, cifar_testset = get_cifar_subset()
    train_data = DataLoader(cifar_trainset, batch_size=args.batch_size, shuffle=True)
    test_data = DataLoader(cifar_testset, batch_size=args.batch_size, shuffle=False)
    # Define optimizer just for the last layer we are finetuning
    optimizer = t.optim.Adam(model.out_layers[-1].parameters(), lr=args.learning_rate)
    # Empty lists
    loss_list = []
    accuracy_list =[]

    # For every epoch:
    for epoch in range(args.epochs):
        epoch_accurate_predictions = 0 # For validating
        pbar = tqdm(train_data)

        # Train - loop for every batch in this epoch
        model.train()
        for imgs, labels in pbar:
            # Move data to device, perform forward pass
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs)
            # Compute loss, perform backward pass
            loss = F.cross_entropy(logits, labels)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            # Update logs & progress bar
            loss_list.append(loss.item())
            pbar.set_postfix(epoch=f"{epoch + 1}/{args.epochs}", loss=f"{loss:.3f}")

        # Test to get accuracy
        model.eval()
        for imgs, labels in test_data:
            imgs, labels = imgs.to(device), labels.to(device)
            with t.inference_mode():
                logits = model(imgs)

            # Calculate accuracy
            # Get max logit, which is treated as the models guess
            predictions = t.argmax(logits, dim=1)
            # Create tensor with 1 if the prediction was correct, 0 if not, for every img
            accuracy_bools = predictions == labels
            # Number of accurate guesses
            epoch_accurate_predictions += accuracy_bools.sum().item()

        # Before completing the epoch
        # Add accuracy percentage to list
        epoch_accuracy = epoch_accurate_predictions / len(cifar_testset)
        accuracy_list.append(epoch_accuracy)

    return loss_list, accuracy_list, model



args = ResNetTrainingArgs(epochs=1)
loss_list, accuracy_list, model = train(args)

In [ ]:
# Display training loss and accuracy
#####################################################
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(8, 5))

ax1.plot(loss_list, color="tab:purple", linewidth=0.8)
ax1.set_xlabel("Batches seen")
ax1.set_ylabel("Cross entropy loss", color="tab:purple")

# one accuracy point at the end of each epoch
batches_per_epoch = len(loss_list) / args.epochs
acc_x = [(i + 1) * batches_per_epoch for i in range(len(accuracy_list))]

ax2 = ax1.twinx()
ax2.plot(acc_x, accuracy_list, color="tab:orange", marker="o")
ax2.set_ylabel("Test accuracy", color="tab:orange")
ax2.set_ylim(0, 1)

plt.title("ResNet feature extraction")
plt.show()

In [ ]:
# Run stats
#####################################################
# The accuracy here is what the import wizard asks for, in percent
final_accuracy = accuracy_list[-1]
best_accuracy = max(accuracy_list)
batches_per_epoch = len(loss_list) // args.epochs
final_epoch_loss = sum(loss_list[-batches_per_epoch:]) / batches_per_epoch

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"final accuracy   {final_accuracy:.2%}")
print(f"best accuracy    {best_accuracy:.2%}  (epoch {accuracy_list.index(best_accuracy) + 1})")
print(f"first loss       {loss_list[0]:.3f}")
print(f"final loss       {final_epoch_loss:.3f}  (mean over the last epoch)")
print()
print(f"epochs           {args.epochs}")
print(f"batch size       {args.batch_size}")
print(f"learning rate    {args.learning_rate}")
print(f"batches seen     {len(loss_list)}  ({batches_per_epoch} per epoch)")
print(f"train images     {batches_per_epoch * args.batch_size:,}")
print()
print(f"parameters       {total_params:,} total, {trainable_params:,} trainable")
print(f"image size       {IMAGE_SIZE}x{IMAGE_SIZE}, ImageNet normalisation")

In [ ]:
# Save model
############################################
t.save(model.state_dict(), "cifar_resnet34.pt")
from google.colab import files; files.download("cifar_resnet34.pt")